In [1]:
import pandas as pd
import numpy as np
from src.data.metrics import get_metrics, exclude_tuning_samples
from src.data.utils import load_data, get_trajectories, get_discretization

In [2]:
timestamps = []
for n_users in [400, 800]:
    # Create a dataset and dataloader
    df, x_features, t_features, _ = load_data(
        dataset="foursquare_TKY",
        n_users=n_users,
    )

    df_quantized = get_discretization(
        df, n_rows=200, shape="hex", aggregation_mode="grow", grow_factor=2
    )

    trajectories = get_trajectories(df_quantized, x_features, t_features)

    x, t, ll, u = zip(*trajectories)
    timestamps.append(pd.Series([ti[0][-1] for ti in t]))

timestamps[1][(timestamps[1] == timestamps[0].iloc[4999])]

8838    2097788.0
dtype: float64

In [2]:
N_TUNE_800 = 8839
N_TUNE_400 = 5000

In [3]:
def highlight_max(s):
    """Highlight maximum value in bold for a Series."""
    is_max = s == s.max()
    return ["font-weight: bold" if v else "" for v in is_max]


def highlight_min(s):
    """Highlight maximum value in bold for a Series."""
    is_min = s == s.min()
    return ["font-weight: bold" if v else "" for v in is_min]

In [4]:
# Replay Methods
df = pd.read_json("../reports/TUL.jsonl", lines=True, orient="records")
df["replay_buffer_type"] = df["replay_buffer_type"].fillna("none")

rep_names = {"recent": "FIFO", "balanced": "Balanced", "random": "Random"}
idcs_replay = df[df["replay_buffer_type"] != "none"].index
df.loc[idcs_replay, "Type"] = "Replay"
df.loc[idcs_replay, "Method"] = df.loc[idcs_replay, "replay_buffer_type"].apply(
    rep_names.get
)

# Replay + MiLES Methods
idcs_replay_miles = df[
    (df["replay_buffer_type"] != "none")
    & (df["embedding_type"] == "lookup_weighted_concat")
].index
df.loc[idcs_replay_miles, "Method"] = df.loc[idcs_replay_miles, "Method"] + " + MiLES"

# Embedding Methods
idcs_emb = df[
    (df["replay_buffer_type"] == "none") & (df["optimizer_cls"] == "Adam")
].index
df.loc[idcs_emb, "Type"] = "Embedding"

In [ ]:
idcs_space2vec = df[~df["embedding_lambda_max"].isna()].index
df.loc[idcs_space2vec, "Method"] = 'Space2Vec'

In [12]:
len(idcs_space2vec)

103

In [ ]:
df = pd.read_json("../reports/TUL.jsonl", lines=True, orient="records")



,dataset,n_users,loc_levels,loc_level,time_levels,model_cls,optimizer_cls,lr,n_hidden,n_layers,...,replay_buffer_type,replay_interval,user_embedding_factor,pred_dists,drift_location,drift_users,pre_drift_lon_limit,method,embedding_lambda_max,embedding_use_poi
1020,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1021,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1022,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1023,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1024,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208,foursquare_TKY,800,1,NaN,1,MainTUL,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,40000.0,1.0
1209,geolife,150,2,NaN,1,T3S,Adam,0.0001,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,40000.0,0.0
1210,foursquare_TKY,800,1,NaN,1,TULVAE,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,40000.0,1.0
1211,foursquare_NYC,800,1,NaN,1,MainTUL,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,40000.0,1.0


In [20]:
df_s2v = df[df["embedding_type"] == "space2vec"]
df_s2v[df_s2v['embedding_lambda_max'].isna()]

,dataset,n_users,loc_levels,loc_level,time_levels,model_cls,optimizer_cls,lr,n_hidden,n_layers,...,replay_buffer_type,replay_interval,user_embedding_factor,pred_dists,drift_location,drift_users,pre_drift_lon_limit,method,embedding_lambda_max,embedding_use_poi
1020,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1021,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1022,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1023,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1024,foursquare_NYC,800,1,NaN,1,BiTULER,Adam,0.0002,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,geolife,150,2,NaN,1,T3S,Adam,0.0001,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1106,geolife,150,2,NaN,1,T3S,Adam,0.0001,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1107,foursquare_TKY,800,2,NaN,1,T3S,Adam,0.0001,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1108,geolife,150,2,NaN,1,T3S,Adam,0.0001,1024,1,...,none,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Replay Methods
df = pd.read_json("../reports/TUL.jsonl", lines=True, orient="records")
df = df[df['embedding_lambda_max'].isna()]
df["replay_buffer_type"] = df["replay_buffer_type"].fillna("none")

rep_names = {"recent": "FIFO", "balanced": "Balanced", "random": "Random"}
idcs_replay = df[df["replay_buffer_type"] != "none"].index
df.loc[idcs_replay, "Type"] = "Replay"
df.loc[idcs_replay, "Method"] = df.loc[idcs_replay, "replay_buffer_type"].apply(
    rep_names.get
)

# Replay + MiLES Methods
idcs_replay_miles = df[
    (df["replay_buffer_type"] != "none")
    & (df["embedding_type"] == "lookup_weighted_concat")
].index
df.loc[idcs_replay_miles, "Method"] = df.loc[idcs_replay_miles, "Method"] + " + MiLES"

# Embedding Methods
idcs_emb = df[
    (df["replay_buffer_type"] == "none") & (df["optimizer_cls"] == "Adam")
].index
df.loc[idcs_emb, "Type"] = "Embedding"
emb_names = {
    "lookup_sum": "Lookup",
    "lookup_concat": "Lookup",
    "lookup_weighted_concat": "MiLES",
    "fourier": "Fourier",
    "linear": "Linear",
    "space2vec": "Space2Vec",
}
df.loc[idcs_emb, "Method"] = df.loc[idcs_emb, "embedding_type"].apply(emb_names.get)

# Adaptive optimizers
idcs_opt = df[df["optimizer_cls"] != "Adam"].index
df.loc[idcs_opt, "Method"] = df.loc[idcs_opt, "optimizer_cls"]

metrics = ["top_1_accuracy", "top_5_accuracy", "macro_f1"]

exclude_tuning_samples(df)
get_metrics(df, metrics)

dfm = (
    df.groupby(["model_cls", "dataset", "Method", "n_users"])[metrics]
    .mean()
    .reset_index()
)

In [6]:
dfstd = (
    df.groupby(["model_cls", "dataset", "Method", "n_users"])[metrics]
    .std()
    .reset_index()
)
(dfstd[(dfstd["n_users"].isin([150, 800]))]["top_5_accuracy"] * 100).quantile(0.9)

np.float64(0.26887803909701236)

In [14]:
dfm.loc[dfm["n_users"] == 75, "n_users"] = 400
dfm.loc[dfm["n_users"] == 150, "n_users"] = 800

In [15]:
dft = dfm.copy()
dft = dft.set_index(["Method", "dataset", "n_users", "model_cls"])
dft = dft.sort_index()
dft *= 100
dft = dft.rename(
    columns={
        "top_1_accuracy": "Acc@1",
        "top_5_accuracy": "Acc@5",
        "macro_f1": "Macro F1",
    }
)

## Make complete table

In [17]:
dfl = dft.xs(800, level=2)
dfl = dfl.pivot_table(index=["model_cls", "Method"], columns="dataset")
dfl.columns = dfl.columns.reorder_levels([1, 0])
dfl = dfl.sort_index(axis=1)
dfl = dfl[
    np.isin(
        dfl.index.get_level_values(0),
        ["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"],
    )
]
dfl = dfl.rename(
    columns={
        "foursquare_NYC": "Foursquare-NYC",
        "foursquare_TKY": "Foursquare-TKY",
        "geolife": "GeoLife",
    }
)
dfl.rename_axis(["Model", "Method"], axis=0, inplace=True)

In [10]:
methods_order = (
    "Lookup",
    "Linear",
    "Fourier",
    "MiLES",
    # "DoG",
    # "AdamHD",
    "Balanced",
    "FIFO",
    "FIFO + MiLES",
)
dfl_table = dfl.reindex(methods_order, level=1)
dfl_table = dfl_table.loc[(slice(None), methods_order), :]
print(
    dfl_table.style.format(precision=2)
    .apply(highlight_max)
    .to_latex(hrules=True, convert_css=True)
)

\begin{tabular}{llrrrrrrrrr}
\toprule
 & dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 &  & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 \\
Model & Method &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[c]{7}{*}{BiTULER} & Lookup & 60.12 & 67.20 & 57.83 & 63.16 & 74.91 & 61.06 & 37.56 & 70.85 & 26.69 \\
 & Linear & 55.75 & 64.45 & 53.08 & 60.63 & 73.45 & 58.20 & 37.28 & 71.13 & 23.33 \\
 & Fourier & 49.28 & 61.75 & 46.07 & 55.61 & 72.33 & 52.64 & 37.35 & 71.39 & 24.92 \\
 & MiLES & 61.61 & \bfseries 70.78 & 59.54 & \bfseries 64.56 & \bfseries 77.76 & \bfseries 62.37 & 46.28 & 77.83 & 32.96 \\
 & Balanced & 60.04 & 66.95 & 57.95 & 63.08 & 74.56 & 60.89 & 36.91 & 67.83 & 25.28 \\
 & FIFO & 59.89 & 66.93 & 57.93 & 62.99 & 74.55 & 60.87 & 36.74 & 68.50 & 25.73 \\
 & FIFO + MiLES & 61.61 & 70.68 & 59.76 & 64.39 & 77.38 & 62.17 & 45.62 & 76.32 & 32.05 \\
\multirow[c]{7}{*}{DeepTUL} & Look

## Get averages over all models

In [23]:
dfa = dfl.reset_index().groupby("Method").mean(numeric_only=True)
dfa = dfa.loc[
    [
        "Lookup",
        "Linear",
        "Fourier",
        "Space2Vec",
        "MiLES",
        # "DoG",
        # "AdamHD",
        "FIFO",
        "Balanced",
        "FIFO + MiLES",
    ]
]
dfa = dfa.reset_index()


def get_type(name):
    if name in ("Lookup", "Fourier", "Space2Vec", "Linear", "MiLES"):
        return "Embedding"
    elif name in ("DoG", "AdamHD"):
        return "Optimizer"
    else:
        return "Replay"


dfa["Type"] = dfa["Method"].apply(get_type)
dfa = dfa.set_index(["Type", "Method"])

print(
    dfa.style.format(precision=2)
    .apply(highlight_max)
    .to_latex(hrules=True, convert_css=True)
)
dfa

\begin{tabular}{llrrrrrrrrr}
\toprule
 & dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 &  & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 \\
Type & Method &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[c]{5}{*}{Embedding} & Lookup & 56.85 & 63.91 & 54.11 & 58.78 & 70.73 & 55.94 & 35.81 & 71.30 & 25.00 \\
 & Linear & 50.74 & 59.21 & 47.51 & 53.48 & 66.25 & 50.16 & 35.02 & 70.62 & 22.66 \\
 & Fourier & 45.31 & 57.79 & 41.68 & 51.18 & 67.72 & 47.49 & 34.54 & 70.56 & 23.26 \\
 & Space2Vec & 55.18 & 65.98 & 52.47 & 58.01 & 73.10 & 54.97 & \bfseries 53.26 & \bfseries 83.99 & \bfseries 35.64 \\
 & MiLES & 58.39 & 67.41 & 55.90 & 60.53 & 73.85 & 57.74 & 43.93 & 77.27 & 30.55 \\
\multirow[c]{3}{*}{Replay} & FIFO & 57.38 & 64.53 & 55.24 & 59.45 & 71.26 & 56.98 & 37.33 & 70.87 & 25.82 \\
 & Balanced & 57.19 & 64.26 & 54.72 & 58.75 & 70.48 & 55.80 & 36.40 & 68.37 & 25.36 \\
 & FIFO + MiLES & \b

/tmp/ipykernel_2560024/3751205645.py:1: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  dfa = dfl.reset_index().groupby("Method").mean(numeric_only=True)


dataset                Foursquare-NYC                       Foursquare-TKY  \
                                Acc@1      Acc@5   Macro F1          Acc@1   
Type      Method                                                             
Embedding Lookup            56.854139  63.912629  54.109849      58.781879   
          Linear            50.741394  59.207532  47.512502      53.476872   
          Fourier           45.313361  57.786708  41.680857      51.183080   
          Space2Vec         55.184096  65.978089  52.471020      58.014321   
          MiLES             58.385987  67.407625  55.900727      60.528108   
Replay    FIFO              57.378549  64.532708  55.239359      59.451238   
          Balanced          57.186340  64.256972  54.716793      58.745313   
          FIFO + MiLES      58.938983  67.883357  56.869384      61.168530   

dataset                                         GeoLife                        
                            Acc@5   Macro F1      Acc@1      Acc@5   Macro F1  
Type      Method                                                               
Embedding Lookup        70.729194  55.938595  35.811565  71.295537  24.995941  
          Linear        66.251253  50.164902  35.020889  70.616792  22.658010  
          Fourier       67.720431  47.493899  34.543230  70.562519  23.258556  
          Space2Vec     73.099061  54.971326  53.258886  83.989301  35.639312  
          MiLES         73.846347  57.739803  43.928130  77.272396  30.552767  
Replay    FIFO          71.259537  56.976025  37.328362  70.871761  25.822770  
          Balanced      70.475848  55.804218  36.401807  68.369581  25.356559  
          FIFO + MiLES  74.193206  58.680471  45.862455  77.494696  31.874878

## Make base model & MiLES tables

In [9]:
df_base = dfl.xs("Lookup", level=1)
df_miles = dfl.xs("MiLES", level=1)
df_diff = df_miles - df_base

df_base = df_base.loc[["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]]
df_diff = df_diff.loc[["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]]

print(
    df_base.style.format(precision=2)
    .apply(highlight_max)
    .to_latex(hrules=True, convert_css=True)
)
print(
    df_diff.style.background_gradient(
        cmap="RdYlGn", vmin=-12, vmax=12, axis=None, text_color_threshold=0.2
    )
    .format("{:+.2f}")
    .to_latex(hrules=True, convert_css=True)
    .replace(r"\color[HTML]{000000}", "")
)

\begin{tabular}{lrrrrrrrrr}
\toprule
dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 \\
Model &  &  &  &  &  &  &  &  &  \\
\midrule
BiTULER & \bfseries 60.12 & \bfseries 67.20 & \bfseries 57.83 & \bfseries 63.16 & \bfseries 74.91 & \bfseries 61.06 & \bfseries 37.56 & 70.85 & 26.69 \\
TULVAE & 59.79 & 66.77 & 57.32 & 55.82 & 66.23 & 51.68 & 37.08 & 70.45 & 25.25 \\
DeepTUL & 58.72 & 65.48 & 56.60 & 61.17 & 72.49 & 59.10 & 36.32 & \bfseries 72.64 & \bfseries 29.82 \\
MainTUL & 55.67 & 62.61 & 53.01 & 59.53 & 71.89 & 57.09 & 34.00 & 70.26 & 21.76 \\
T3S & 52.98 & 60.28 & 49.50 & 56.41 & 69.26 & 53.24 & 35.25 & 71.11 & 21.52 \\
TULHOR & 53.85 & 61.13 & 50.40 & 56.61 & 69.61 & 53.45 & 34.65 & 72.46 & 24.92 \\
\bottomrule
\end{tabular}

\begin{tabular}{lrrrrrrrrr}
\toprule
dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursq

In [10]:
df_diff = (df_miles / df_base) - 1
df_diff = df_diff.loc[["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]]

In [18]:
print((df_diff * 100).style.format("{:+.2f}").to_latex())

\begin{tabular}{lrrrrrrrrr}
dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 \\
Model &  &  &  &  &  &  &  &  &  \\
BiTULER & +2.47 & +5.33 & +2.97 & +2.22 & +3.81 & +2.14 & +23.20 & +9.85 & +23.46 \\
TULVAE & +2.99 & +5.59 & +3.56 & +5.49 & +6.69 & +6.46 & +21.61 & +9.32 & +18.76 \\
DeepTUL & +1.80 & +4.64 & +2.22 & +1.37 & +3.27 & +1.30 & +23.66 & +8.68 & +20.02 \\
MainTUL & +2.59 & +6.92 & +2.87 & +2.73 & +4.81 & +2.55 & +24.10 & +8.80 & +25.29 \\
T3S & +3.23 & +5.20 & +4.24 & +3.01 & +4.03 & +3.60 & +19.81 & +6.75 & +24.07 \\
TULHOR & +3.17 & +5.17 & +4.18 & +3.27 & +4.03 & +3.80 & +23.66 & +6.94 & +22.81 \\
\end{tabular}



In [21]:
(
    df_diff.style.background_gradient(
        cmap="RdYlGn", vmin=-12, vmax=12, axis=None, text_color_threshold=0.2
    )
    .format("{:+.2f}")
)

In [13]:
# Calculate relative difference between df_miles and df_base
rel_diff = ((df_miles - df_base) / df_base).xs("Acc@1", axis=1, level=1)
rel_diff

dataset,Foursquare-NYC,Foursquare-TKY,GeoLife
Model,,,
BiTULER,0.024742,0.022220,0.232037
DeepTUL,0.018033,0.013721,0.236622
MainTUL,0.025882,0.027294,0.240967
T3S,0.032261,0.030062,0.198113
TULHOR,0.031697,0.032669,0.236603
TULVAE,0.029902,0.054904,0.216106


In [14]:
# Make tables for lower user counts

df4 = dft.xs(400, level=2)
df4 = df4.pivot_table(index=["model_cls", "Method"], columns="dataset")
df4.columns = df4.columns.reorder_levels([1, 0])
df4 = df4.sort_index(axis=1)
df4 = df4[
    np.isin(
        df4.index.get_level_values(0),
        ["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"],
    )
]
df4 = df4.rename(
    columns={
        "foursquare_NYC": "Foursquare-NYC",
        "foursquare_TKY": "Foursquare-TKY",
        "geolife": "GeoLife",
    }
)
df4.rename_axis(["Model", "Method"], axis=0, inplace=True)

df_base = df4.xs("Lookup", level=1)
df_miles = df4.xs("MiLES", level=1)
df_diff = df_miles - df_base

df_base = df_base.loc[["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]]
df_diff = df_diff.loc[["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]]

print(
    df_base.style.format(precision=2)
    .apply(highlight_max)
    .to_latex(hrules=True, convert_css=True)
)
print(
    df_diff.style.background_gradient(
        cmap="RdYlGn", vmin=-12, vmax=12, axis=None, text_color_threshold=0.2
    )
    .format("{:+.2f}")
    .to_latex(hrules=True, convert_css=True)
    .replace(r"\color[HTML]{000000}", "")
)

\begin{tabular}{lrrrrrrrrr}
\toprule
dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 \\
Model &  &  &  &  &  &  &  &  &  \\
\midrule
BiTULER & \bfseries 69.95 & \bfseries 75.85 & \bfseries 68.59 & \bfseries 73.29 & \bfseries 84.06 & \bfseries 72.03 & \bfseries 39.98 & 74.16 & 32.95 \\
TULVAE & 69.77 & 75.72 & 68.37 & 66.86 & 77.42 & 64.05 & 39.40 & 73.82 & 31.57 \\
DeepTUL & 68.93 & 74.77 & 67.62 & 71.65 & 82.47 & 70.42 & 38.44 & 75.15 & \bfseries 33.42 \\
MainTUL & 66.28 & 72.41 & 64.86 & 70.69 & 82.30 & 69.28 & 35.89 & 73.02 & 27.87 \\
T3S & 63.84 & 70.42 & 61.84 & 66.77 & 79.41 & 64.89 & 37.79 & 74.68 & 29.06 \\
TULHOR & 64.60 & 71.08 & 62.69 & 67.21 & 79.57 & 65.36 & 37.22 & \bfseries 75.43 & 30.87 \\
\bottomrule
\end{tabular}

\begin{tabular}{lrrrrrrrrr}
\toprule
dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursq

In [15]:
df_base.style.format(precision=2).apply(highlight_max)

In [ ]:
df_base = df4.xs("Lookup", level=1)
df_miles = df4.xs("MiLES", level=1)
df_diff = df_miles - df_base

df_base = df_base.loc[["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]]
df_diff = df_diff.loc[["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]]

df_base = pd.concat({"Default": df_base}, names=["Embedding", "Model"])
df_diff = pd.concat({r"$\Delta$ MiLES": df_diff}, names=["Embedding", "Model"])

print(
    dfa.style.format(precision=2)
    .apply(highlight_max)
    .to_latex(hrules=True, convert_css=True)
)
print(
    df_diff.style.background_gradient(
        cmap="RdYlGn", vmin=-12, vmax=12, axis=None, text_color_threshold=0.2
    )
    .format("{:+.2f}")
    .to_latex(hrules=True, convert_css=True)
    .replace(r"\color[HTML]{000000}", "")
)

\begin{tabular}{llrrrrrrrrr}
\toprule
 & dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 &  & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 & Acc@1 & Acc@5 & Macro F1 \\
Type & Method &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[c]{4}{*}{Embedding} & Lookup & 56.85 & 63.91 & 54.11 & 58.78 & 70.73 & 55.94 & 35.81 & 71.30 & 25.00 \\
 & Linear & 50.74 & 59.21 & 47.51 & 53.48 & 66.25 & 50.16 & 35.02 & 70.62 & 22.66 \\
 & Fourier & 45.31 & 57.79 & 41.68 & 51.18 & 67.72 & 47.49 & 34.54 & 70.56 & 23.26 \\
 & MiLES & 58.39 & 67.41 & 55.90 & 60.53 & 73.85 & 57.74 & 43.93 & 77.27 & 30.55 \\
\multirow[c]{3}{*}{Replay} & FIFO & 57.38 & 64.53 & 55.24 & 59.45 & 71.26 & 56.98 & 37.33 & 70.87 & 25.82 \\
 & Balanced & 57.19 & 64.26 & 54.72 & 58.75 & 70.48 & 55.80 & 36.40 & 68.37 & 25.36 \\
 & FIFO + MiLES & \bfseries 58.94 & \bfseries 67.88 & \bfseries 56.87 & \bfseries 61.17 & \bfseries 74.19 & \bfseries 58.68 & \bfseries 45

## Make runtime plots

In [ ]:
from src.data.utils import get_dataloader

results = []
for dataset in ["foursquare_NYC", "foursquare_TKY", "geolife"]:
    n_users = 150 if dataset == "geolife" else 800
    discretization_rows = 800 if dataset == "geolife" else 200
    _, n_locs, _ = get_dataloader(
        dataset,
        discretization_rows=discretization_rows,
        aggregation_mode="grow",
        n_users=n_users,
    )
    for n_levels in [1, 4]:
        embedding_dim_loc = 1024
        weight_factor = 2
        loc_level_weights = np.array(
            [weight_factor**-i for i in (range(n_levels))], dtype=float
        )
        loc_level_weights /= loc_level_weights.sum()
        loc_level_dims = (loc_level_weights * embedding_dim_loc).astype(int)
        loc_level_dims[0] = embedding_dim_loc - loc_level_dims[1:].sum()
        loc_level_dims
        n_params = np.array(n_locs[:n_levels]) @ loc_level_dims
        results.append(
            {
                "dataset": dataset,
                "embedding_type": "lookup_weighted_concat"
                if n_levels == 4
                else "lookup_concat",
                "#Parameters": n_params,
            }
        )
df_params = pd.DataFrame.from_records(results)

In [44]:
df = pd.read_json("../reports/wall_time.jsonl", orient="records", lines=True)
df = df[["dataset", "embedding_type", "seed", "runtime"]]
df = pd.merge(df, df_params, on=['dataset', 'embedding_type'])
df["dataset"] = df["dataset"].replace(
    {
        "foursquare_NYC": "Foursquare-NYC",
        "foursquare_TKY": "Foursquare-TKY",
        "geolife": "GeoLife",
    }
)
df["embedding_type"] = df["embedding_type"].replace(
    {"lookup_concat": "Base", "lookup_weighted_concat": "MiLES"}
)
n_samples = {
    "Foursquare-NYC": 61_218,
    "Foursquare-TKY": 70_007,
    "GeoLife": 25_611,
}
df["n_samples"] = df["dataset"].apply(n_samples.get)
df["runtime"] = df["runtime"] / df["n_samples"] * 1000
df["#Parameters"] = df["#Parameters"] / 1e6
dfm = df.groupby(["dataset", "embedding_type"])[["runtime", "#Parameters"]].mean()
dft = dfm.pivot_table(columns='dataset', index='embedding_type')
dft.columns = dft.columns.reorder_levels([1, 0])
dft = dft.sort_index(axis=1)
print(dft.style.format("{:.2f}").to_latex(hrules=True))

\begin{tabular}{lrrrrrr}
\toprule
dataset & \multicolumn{2}{r}{Foursquare-NYC} & \multicolumn{2}{r}{Foursquare-TKY} & \multicolumn{2}{r}{GeoLife} \\
 & #Parameters & runtime & #Parameters & runtime & #Parameters & runtime \\
embedding_type &  &  &  &  &  &  \\
\midrule
Base & 35.21 & 7.99 & 39.13 & 8.50 & 4.74 & 11.27 \\
MiLES & 21.97 & 7.38 & 24.32 & 7.81 & 3.28 & 11.66 \\
\bottomrule
\end{tabular}



## Destination Prediction

In [9]:
df = pd.read_json("../reports/DP.jsonl", orient="records", lines=True)

In [10]:
rep_names = {"recent": "FIFO", "balanced": "Balanced", "random": "Random"}
idcs_replay = df[df["replay_buffer_type"] != "none"].index
df.loc[idcs_replay, "Type"] = "Replay"
df.loc[idcs_replay, "Method"] = df.loc[idcs_replay, "replay_buffer_type"].apply(
    rep_names.get
)

idcs_replay_miles = df[
    (df["replay_buffer_type"] != "none")
    & (df["embedding_type"] == "lookup_weighted_concat")
].index
df.loc[idcs_replay_miles, "Method"] = df.loc[idcs_replay_miles, "Method"] + " + MiLES"

idcs_emb = df[df["replay_buffer_type"] == "none"].index
df.loc[idcs_emb, "Type"] = "Embedding"
emb_names = {
    "lookup_concat": "Lookup",
    "lookup_weighted_concat": "MiLES",
    "fourier": "Fourier",
    "linear": "Linear",
}
df.loc[idcs_emb, "Method"] = df.loc[idcs_emb, "embedding_type"].apply(emb_names.get)


exclude_tuning_samples(df, ["pred_dists"])
metrics = ["Mean", "P50", "P90"]


df["Mean"] = df["pred_dists"].apply(np.mean)
df["P50"] = df["pred_dists"].apply(np.median)
df["P90"] = df["pred_dists"].apply(lambda x: np.quantile(x, 0.95))

In [13]:
dfm = dfn.groupby(["dataset", "model_cls", "Method"])[metrics].agg(['mean', 'std'])

In [14]:
dfm

Mean                     P50  \
                                          mean        std         mean   
dataset        model_cls   Method                                        
foursquare_NYC BiLSTM      Lookup  7213.197372  59.232726  5369.533129   
                           MiLES   6492.575901  28.380952  4724.307703   
               CNN         Lookup  6508.833505  24.043762  4809.023754   
                           MiLES   6379.382509  30.446152  4626.886333   
               MLP         Lookup  7803.654386  44.756677  6139.989953   
                           MiLES   7257.661873  35.489362  5624.307278   
               Transformer Lookup  8901.471527  57.839177  7261.845634   
                           MiLES   7886.873568  56.541313  6293.359788   
foursquare_TKY BiLSTM      Lookup  6974.860384  34.447899  5600.497313   
                           MiLES   6735.270194  20.302469  5306.193413   
               CNN         Lookup  6567.470287  25.151549  5211.663284   
                           MiLES   6614.700170  26.111250  5252.713200   
               MLP         Lookup  7530.181925  45.601917  6233.369589   
                           MiLES   7365.537758  19.649470  6055.779292   
               Transformer Lookup  8111.652452  49.819044  6928.861075   
                           MiLES   7632.020537  39.035751  6421.316348   
geolife_dp     BiLSTM      Lookup  5957.345734  18.159360  4285.185213   
                           MiLES   5871.243406  11.781275  4210.370891   
               CNN         Lookup  6326.693532   8.363932  4563.875901   
                           MiLES   6131.274004  27.952399  4364.743759   
               MLP         Lookup  7117.491715  35.244691  5394.193162   
                           MiLES   6956.899772  23.355455  5228.303029   
               Transformer Lookup  7542.231447  41.339910  5811.230164   
                           MiLES   7454.011407  63.938639  5719.618510   

                                                       P90              
                                         std          mean         std  
dataset        model_cls   Method                                       
foursquare_NYC BiLSTM      Lookup  49.307536  20032.473120  209.380765  
                           MiLES   32.571750  18476.288475   75.700717  
               CNN         Lookup  17.174698  18275.266570  103.528008  
                           MiLES   35.798016  18168.451121   49.988075  
               MLP         Lookup  57.723984  20079.536503  148.239202  
                           MiLES   40.322138  19088.604569   98.320532  
               Transformer Lookup  38.733004  21808.358086  222.585354  
                           MiLES   80.210268  19921.951222  134.796760  
foursquare_TKY BiLSTM      Lookup  29.636301  17677.699237  137.304611  
                           MiLES   24.994204  17439.955444   44.220465  
               CNN         Lookup  26.816875  16858.343567   18.791504  
                           MiLES   22.636176  16984.171649  115.059518  
               MLP         Lookup  60.279107  18091.602876   39.381053  
                           MiLES   33.718602  17876.803863   41.837411  
               Transformer Lookup  48.875820  18578.166559  141.122439  
                           MiLES   47.134460  17988.235006  115.817171  
geolife_dp     BiLSTM      Lookup  20.223274  16897.773833  127.371053  
                           MiLES   11.798606  16554.495901   82.730111  
               CNN         Lookup  32.069044  17868.930307   76.896453  
                           MiLES   26.330050  17585.845013  160.751674  
               MLP         Lookup  39.212975  19098.139356  170.925431  
                           MiLES   45.330225  18849.234985  134.035105  
               Transformer Lookup  24.463762  19966.266557  217.806288  
                           MiLES   79.477371  19792.439739  150.051042

In [11]:
# Make MiLES vs. standard embeddings table
dfn = df[df["Method"].isin(["Lookup", "MiLES"])]
dfm = dfn.groupby(["dataset", "model_cls", "Method"])[metrics].mean()
dft = dfm.pivot_table(index=["Method", "model_cls"], columns="dataset")
dft = dft.reorder_levels([1, 0], axis=1)
dft = dft.sort_index(axis=1)

dft = dft.rename(
    columns={
        "foursquare_NYC": "Foursquare-NYC",
        "foursquare_TKY": "Foursquare-TKY",
        "geolife_dp": "GeoLife",
    }
)
dft /= 1000

df_base = dft.xs("Lookup", level=0)
df_miles = dft.xs("MiLES", level=0)

In [12]:
df_diff = df_miles - df_base
df_base = pd.concat({"Default": df_base}, names=["Embedding", "Model"])
df_diff = pd.concat({r"$\Delta$ MiLES": df_diff}, names=["Embedding", "Model"])
print(
    df_base.style.apply(highlight_min)
    .format("{:.2f}")
    .to_latex(hrules=True, convert_css=True)
)
print(
    df_diff.style.background_gradient(
        cmap="RdYlGn_r", vmin=-2.5, vmax=2.5, axis=None, text_color_threshold=0.0
    )
    .format("{:-.2f}")
    .to_latex(hrules=True, convert_css=True)
    .replace(r"\color[HTML]{000000}", "")
)

\begin{tabular}{llrrrrrrrrr}
\toprule
 & dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 &  & Mean & P50 & P90 & Mean & P50 & P90 & Mean & P50 & P90 \\
Embedding & Model &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[c]{4}{*}{Default} & BiLSTM & 7.21 & 5.37 & 20.03 & 6.97 & 5.60 & 17.68 & \bfseries 5.96 & \bfseries 4.29 & \bfseries 16.90 \\
 & CNN & \bfseries 6.51 & \bfseries 4.81 & \bfseries 18.28 & \bfseries 6.57 & \bfseries 5.21 & \bfseries 16.86 & 6.33 & 4.56 & 17.87 \\
 & MLP & 7.80 & 6.14 & 20.08 & 7.53 & 6.23 & 18.09 & 7.12 & 5.39 & 19.10 \\
 & Transformer & 8.90 & 7.26 & 21.81 & 8.11 & 6.93 & 18.58 & 7.54 & 5.81 & 19.97 \\
\bottomrule
\end{tabular}

\begin{tabular}{llrrrrrrrrr}
\toprule
 & dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 &  & Mean & P50 & P90 & Mean & P50 & P90 & Mean & P50 & P90 \\
Embedding & Model &  &  &  &  &  &  &  &  & 

In [ ]:
# Make method tables
dfm = df.groupby(["dataset", "Method", "Type"])[metrics].mean()
dft = dfm.pivot_table(index=["Type", "Method"], columns="dataset")
dft = dft.reorder_levels([1, 0], axis=1)
dft = dft.sort_index(axis=1)

dft = dft.rename(
    columns={
        "foursquare_NYC": "Foursquare-NYC",
        "foursquare_TKY": "Foursquare-TKY",
        "geolife_dp": "GeoLife",
    }
)
dft = dft.loc[
    (
        slice(None),
        ["Lookup", "Linear", "Fourier", "MiLES", "FIFO", "Random", "FIFO + MiLES"],
    ),
    :,
]
dft = dft / 1000

print(
    dft.style.apply(highlight_min)
    .format("{:.2f}")
    .to_latex(hrules=True, convert_css=True)
)
dft


\begin{tabular}{llrrrrrrrrr}
\toprule
 & dataset & \multicolumn{3}{r}{Foursquare-NYC} & \multicolumn{3}{r}{Foursquare-TKY} & \multicolumn{3}{r}{GeoLife} \\
 &  & Mean & P50 & P90 & Mean & P50 & P90 & Mean & P50 & P90 \\
Type & Method &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[c]{4}{*}{Embedding} & Lookup & 7.61 & 5.90 & 20.05 & 7.30 & 5.99 & 17.80 & 6.74 & 5.01 & 18.46 \\
 & Linear & 7.13 & 5.56 & \bfseries 18.63 & 7.20 & 5.92 & \bfseries 17.55 & 6.83 & 5.12 & 18.53 \\
 & Fourier & 7.00 & 5.34 & 18.88 & 7.14 & 5.80 & 17.73 & 6.69 & 4.96 & 18.41 \\
 & MiLES & 7.00 & 5.32 & 18.91 & \bfseries 7.09 & 5.76 & 17.57 & 6.60 & 4.88 & 18.20 \\
\multirow[c]{3}{*}{Replay} & FIFO & 7.39 & 5.52 & 20.35 & 7.30 & 5.82 & 18.64 & 6.37 & 4.50 & 18.33 \\
 & Random & 7.39 & 5.50 & 20.41 & 7.31 & 5.82 & 18.62 & 6.31 & 4.43 & 18.27 \\
 & FIFO + MiLES & \bfseries 6.82 & \bfseries 4.97 & 19.35 & 7.10 & \bfseries 5.56 & 18.45 & \bfseries 6.09 & \bfseries 4.26 & \bfseries 17.67 \\
\bottomrule
\end{tabular}

dataset                Foursquare-NYC                      Foursquare-TKY  \
                                 Mean       P50        P90           Mean   
Type      Method                                                            
Embedding Lookup             7.606789  5.895098  20.048909       7.296041   
          Linear             7.125737  5.560730  18.630216       7.195392   
          Fourier            7.004953  5.338640  18.877713       7.141531   
          MiLES              7.004123  5.317215  18.913824       7.086882   
Replay    FIFO               7.388764  5.516375  20.345076       7.303894   
          Random             7.391077  5.497257  20.407835       7.312886   
          FIFO + MiLES       6.820065  4.965989  19.345344       7.095598   

dataset                                       GeoLife                       
                             P50        P90      Mean       P50        P90  
Type      Method                                                            
Embedding Lookup        5.993598  17.801453  6.735941  5.013621  18.457778  
          Linear        5.917385  17.554804  6.831688  5.124819  18.529520  
          Fourier       5.795929  17.733983  6.688537  4.955192  18.411400  
          MiLES         5.759001  17.572291  6.603357  4.880759  18.195504  
Replay    FIFO          5.818861  18.637999  6.374675  4.496473  18.332780  
          Random        5.822976  18.623807  6.313909  4.430349  18.265865  
          FIFO + MiLES  5.561875  18.446715  6.090971  4.255812  17.669979

## Appendix Tables

Migrated from `tools/make_review_tables.py` (previously a standalone script); kept verbatim so the emitted LaTeX is unchanged.

In [ ]:
from functools import lru_cache
from pathlib import Path

from scipy.stats import wilcoxon

REPORTS = Path("../reports")
PUB = Path("../pub")


MODELS = ["BiTULER", "TULVAE", "DeepTUL", "MainTUL", "T3S", "TULHOR"]
# (label, dataset, n_users, discretization_rows) for the main experiment.
DATASETS = [
    ("Foursquare-NYC", "foursquare_NYC", 800, 200),
    ("Foursquare-TKY", "foursquare_TKY", 800, 200),
    ("GeoLife", "geolife", 150, 800),
]
# Both user-count variants per dataset, used for the per-model significance test.
DATASET_VARIANTS = {
    "foursquare_NYC": [400, 800],
    "foursquare_TKY": [400, 800],
    "geolife": [75, 150],
}
DATASET_LABELS = {
    "foursquare_NYC": "Foursquare-NYC",
    "foursquare_TKY": "Foursquare-TKY",
    "geolife": "GeoLife",
}
METRIC_LABELS = ["Acc@1", "Acc@5", "F1"]
METRICS = ["top_1_accuracy", "top_5_accuracy", "macro_f1"]
METRIC_TO_LABEL = dict(zip(METRICS, METRIC_LABELS))
# MiLES main-experiment embedding hyperparameters.
LOC_LEVELS = 4
GROW_FACTOR = 2
WEIGHT_FACTOR = 2
EMB_DIM = 1024


In [ ]:
@lru_cache(maxsize=1)
def _load_main_results():
    """Load the main online results using the same pandas flow as the notebooks."""
    df = pd.read_json(REPORTS / "TUL.jsonl", lines=True, orient="records")
    mask = (
        (df["replay_buffer_type"].isna() | (df["replay_buffer_type"] == "none"))
        & (df["optimizer_cls"] == "Adam")
        & df["labels"].notna()
    )
    df = df.loc[mask].copy()
    df["model_name"] = _model_names(df)
    df = df[df["model_name"].notna()].copy()
    exclude_tuning_samples(df)
    get_metrics(df, METRICS)
    df[METRICS] *= 100
    return df


def _model_names(df):
    model_cls = df["model_cls"].astype(str)
    model_name = pd.Series(pd.NA, index=df.index, dtype="object")
    for model in MODELS:
        model_name = model_name.mask(model_cls.str.contains(model, regex=False), model)
    return model_name


def _fmt_pvalue(p):
    """Format a p-value with tab:wilcoxon-style significance shading."""
    if p < 0.001:
        return r"{\cellcolor[HTML]{A7D96B}} $<$0.001"
    if p < 0.01:
        return rf"{{\cellcolor[HTML]{{BDE379}}}} {p:.3f}"
    if p < 0.05:
        return rf"{{\cellcolor[HTML]{{E8F59F}}}} {p:.3f}"
    return f"{p:.3f}"


def wilcoxon_default_table():
    df = _load_main_results()
    variant_rows = [
        {"dataset": dataset, "n_users": n_users}
        for dataset, variants in DATASET_VARIANTS.items()
        for n_users in variants
    ]
    variants = pd.DataFrame.from_records(variant_rows)
    df = df.merge(variants, on=["dataset", "n_users"])

    index_cols = ["dataset", "n_users", "model_name", "seed"]
    default = df[df["embedding_type"] == "lookup_concat"][index_cols + METRICS]
    miles = df[df["embedding_type"] == "lookup_weighted_concat"][index_cols + METRICS]
    paired = miles.merge(default, on=index_cols, suffixes=("_miles", "_default"))

    records = []
    for (model, dataset), df_pair in paired.groupby(["model_name", "dataset"]):
        for metric in METRICS:
            diffs = df_pair[f"{metric}_miles"] - df_pair[f"{metric}_default"]
            records.append(
                {
                    "model_name": model,
                    "dataset": dataset,
                    "metric": METRIC_TO_LABEL[metric],
                    "pvalue": wilcoxon(diffs, alternative="greater").pvalue,
                    "n_obs": len(diffs),
                }
            )
    df_wilc = pd.DataFrame.from_records(records)
    table = df_wilc.pivot_table(
        index="model_name", columns=["dataset", "metric"], values="pvalue", aggfunc="first"
    )
    table = table.rename(columns=DATASET_LABELS, level=0)
    table = table.reindex(index=MODELS)
    table = table.reindex(
        columns=pd.MultiIndex.from_product(
            [[DATASET_LABELS[dataset] for dataset in DATASET_VARIANTS], METRIC_LABELS]
        )
    )
    table = table.map(_fmt_pvalue)


    table.index.name = "Model"
    table.columns.names = ["Dataset", ""]
    return table.to_latex(
        escape=False,
        column_format="lrrrrrrrrr",
        multicolumn=True,
        multicolumn_format="c",
        label="tab:wilcoxon_default",
        index_names=False,
        position="hbt",
    )


In [ ]:
tex_wilcoxon = wilcoxon_default_table()
(PUB / "_wilcoxon_default_table.tex").write_text(tex_wilcoxon + "\n")
print(tex_wilcoxon)


In [ ]:
def variance_table():
    df = _load_main_results()
    main_variants = pd.DataFrame.from_records(
        [
            {"dataset_label": label, "dataset": dataset, "n_users": n_users}
            for label, dataset, n_users, _ in DATASETS
        ]
    )
    df = df.merge(main_variants, on=["dataset", "n_users"])
    df = df[df["embedding_type"].isin(["lookup_concat", "lookup_weighted_concat"])].copy()

    index_cols = ["dataset", "dataset_label", "n_users", "model_name", "seed"]
    default = df[df["embedding_type"] == "lookup_concat"][index_cols + METRICS]
    miles = df[df["embedding_type"] == "lookup_weighted_concat"][index_cols + METRICS]
    paired = miles.merge(default, on=index_cols, suffixes=("_miles", "_default"))
    for metric in METRICS:
        paired[metric] = paired[f"{metric}_miles"] - paired[f"{metric}_default"]

    summary = (
        paired.groupby(["model_name", "dataset", "dataset_label"])[METRICS]
        .agg(["mean", lambda s: s.std(ddof=0)])
        .rename(columns={"<lambda_0>": "std"})
    )

    records = []
    for model in MODELS:
        record = {"Model": model}
        for _, dataset, _, _ in DATASETS:
            row = summary.loc[(model, dataset, DATASET_LABELS[dataset])]
            for metric in METRICS:
                mean = row[(metric, "mean")]
                std = row[(metric, "std")]
                record[(DATASET_LABELS[dataset], METRIC_TO_LABEL[metric])] = (
                    f"{mean:.2f}\\,$\\pm$\\,{std:.2f}"
                )
        records.append(record)
    table = pd.DataFrame.from_records(records).set_index("Model")
    table.columns = pd.MultiIndex.from_tuples(table.columns, names=["Dataset", ""])
    return table.to_latex(
        escape=False,
        column_format="lccccccccc",
        multicolumn=True,
        multicolumn_format="c",
        label="tab:models_variance",
        index_names=False,
        position="hbt",
    )


In [ ]:
tex_variance = variance_table()
(PUB / "_variance_table.tex").write_text(tex_variance + "\n")
print(tex_variance)


In [ ]:
def param_breakdown_table():
    weights = np.array([WEIGHT_FACTOR ** -i for i in range(LOC_LEVELS)], dtype=float)
    weights /= weights.sum()
    dims = (weights * EMB_DIM).astype(int)
    dims[0] = EMB_DIM - dims[1:].sum()

    records = []
    for label, dataset, n_users, rows_d in DATASETS:
        df, x_features, _, _ = load_data(dataset, n_users)
        dfd = get_discretization(
            df, n_rows=rows_d, shape="hex", aggregation_mode="grow", grow_factor=GROW_FACTOR
        )
        feats = x_features[:LOC_LEVELS]
        for level, (feature, dim) in enumerate(zip(feats, dims)):
            locations = int(dfd[feature].nunique()) + 1
            records.append(
                {
                    "dataset": label,
                    "level": level,
                    "feature": feature,
                    "locations": locations,
                    "dim": int(dim),
                    "parameters": int(locations * dim),
                }
            )
    table = pd.DataFrame.from_records(records)
    table["Level"] = table["level"].astype(str) + " (" + table["feature"] + ")"
    table = table[["dataset", "Level", "locations", "dim", "parameters"]].rename(
        columns={
            "dataset": "Dataset",
            "locations": "Locations",
            "dim": "Dim",
            "parameters": "Parameters",
        }
    )
    total_rows = []
    for label in table["Dataset"].drop_duplicates():
        df_dataset = table[table["Dataset"] == label]
        miles_total = df_dataset["Parameters"].sum()
        default_locations = int(df_dataset.iloc[0]["Locations"])
        total_rows += [
            {
                "Dataset": label,
                "Level": "MiLES total",
                "Locations": "",
                "Dim": int(dims.sum()),
                "Parameters": miles_total,
            },
            {
                "Dataset": label,
                "Level": "Default",
                "Locations": default_locations,
                "Dim": EMB_DIM,
                "Parameters": default_locations * EMB_DIM,
            },
        ]
    table = pd.concat([table, pd.DataFrame.from_records(total_rows)], ignore_index=True)
    table["Dataset"] = pd.Categorical(
        table["Dataset"], [label for label, *_ in DATASETS], ordered=True
    )
    level_order = (
        table.loc[~table["Level"].isin(["MiLES total", "Default"]), "Level"].drop_duplicates().tolist()
        + ["MiLES total", "Default"]
    )
    table["Level"] = pd.Categorical(table["Level"], level_order, ordered=True)
    table = table.sort_values(["Dataset", "Level"]).reset_index(drop=True)
    table["Dataset"] = table["Dataset"].astype(str)
    table["Locations"] = table["Locations"].map(lambda x: "" if x == "" else f"{int(x):,}")
    table["Parameters"] = table["Parameters"].map(lambda x: f"{int(x):,}")

    return table.to_latex(
        index=False,
        escape=False,
        column_format="llrrr",
        label="tab:param_breakdown",
        position="hbt",
    )


In [ ]:
tex_param_breakdown = param_breakdown_table()
(PUB / "_param_table.tex").write_text(tex_param_breakdown + "\n")
print(tex_param_breakdown)


In [ ]:
# Level learning-rate experiment (NYC variants, seeds 0-4).
LEVEL_LR_VARIANTS = [400, 800]
LEVEL_LR_UNIFORM = 1.45
LEVEL_LR_MATCHED = [1.44, 1.36, 1.45, 1.55]
# Total-dimension budget sweep.
BUDGET_FACTORS = [0.125, 0.25, 0.5, 1.0, 2.0]
BUDGET_METHODS = [
    ("lookup_weighted_concat", "MiLES"),
    ("lookup_concat", "POI lookup"),
    ("fourier", "Fourier"),
]


def _compute_metrics(df):
    """Exclude tuning samples and compute percentage metrics on a result frame."""
    df = df[df["labels"].notna()].copy().reset_index(drop=True)
    exclude_tuning_samples(df)
    get_metrics(df, METRICS)
    df[METRICS] = df[METRICS] * 100
    return df


def _nyc800_seed_means(df, seeds=range(5)):
    df = df[df["seed"].isin(seeds)].drop_duplicates(subset=["seed"], keep="last")
    m = _compute_metrics(df)
    return m[METRICS].mean(), m[METRICS].std()


def level_lr_table():
    # A. Full MiLES (learnable w_l) from the main experiment.
    tul = pd.read_json(REPORTS / "TUL.jsonl", lines=True)
    full = tul[
        (tul["dataset"] == "foursquare_NYC")
        & tul["model_cls"].astype(str).str.contains("BiTULER")
        & (tul["optimizer_cls"] == "Adam")
        & (tul["embedding_type"] == "lookup_weighted_concat")
        & (tul["n_users"] == 800)
    ]
    if "replay_buffer_type" in full:
        full = full[full["replay_buffer_type"].isna() | (full["replay_buffer_type"] == "none")]
    # B. -WL (no learnable weights) from the ablation study.
    ab = pd.read_json(REPORTS / "ablation_all.jsonl", lines=True)
    no_wl = ab[
        (ab["dataset"] == "foursquare_NYC")
        & (ab["model_cls"] == "BiTULER")
        & (ab["embedding_type"] == "lookup_concat")
        & (ab["embedding_weight_factor"] == 2)
        & (ab["loc_levels"] == 4)
        & ab["loc_level"].isna()
        & (ab["n_users"] == 800)
    ]
    # C. -WL with a per-step per-level learning rate equal to the recorded w_l(t).
    schedule = pd.read_json(REPORTS / "level_lr_schedule.jsonl", lines=True)

    conditions = [
        ("Full MiLES (learnable $w_l$)", full),
        ("$-$WL (no level weights)", no_wl),
        (r"$-$WL $+$ per-step LR $\propto w_l(t)$", schedule),
    ]
    records = []
    for name, df in conditions:
        mean, std = _nyc800_seed_means(df)
        record = {"Embedding": name}
        for metric in METRICS:
            record[METRIC_TO_LABEL[metric]] = (
                f"{mean[metric]:.2f}\\,$\\pm$\\,{std[metric]:.2f}"
            )
        records.append(record)
    table = pd.DataFrame.from_records(records).set_index("Embedding")


    return table.to_latex(
        escape=False,
        column_format="lccc",
        label="tab:level_lr",
        index_names=False,
        position="hbt",
    )


In [ ]:
tex_level_lr = level_lr_table()
(PUB / "_level_lr_table.tex").write_text(tex_level_lr + "\n")
print(tex_level_lr)


In [ ]:
def dim_budget_table():
    df = pd.read_json(REPORTS / "dim_budget_sweep.jsonl", lines=True)
    df = df[df["dataset"] == "foursquare_NYC"]
    df = _compute_metrics(df)
    df["dim"] = (df["loc_embedding_factor"] * EMB_DIM).round().astype(int)
    means = df.groupby(["embedding_type", "dim"])[METRICS].mean()
    dims = sorted(int(f * EMB_DIM) for f in BUDGET_FACTORS)


    lines = [
        r"\begin{table}[hbt]",
        r"\centering",
        r"\label{tab:dim_budget}",
        r"\begin{tabular}{ll" + "r" * len(dims) + "}",
        r"\toprule",
        r"Metric & Embedding & " + " & ".join(f"$d={d}$" for d in dims) + r" \\",
        r"\midrule",
    ]
    for mi, metric in enumerate(METRICS):
        for j, (emb, name) in enumerate(BUDGET_METHODS):
            cells = []
            for d in dims:
                val = means.loc[(emb, d), metric]
                colmax = max(means.loc[(e, d), metric] for e, _ in BUDGET_METHODS)
                cell = f"{val:.2f}"
                if abs(val - colmax) < 1e-9:
                    cell = r"\bfseries " + cell
                cells.append(cell)
            head = (
                rf"\multirow{{{len(BUDGET_METHODS)}}}{{*}}{{{METRIC_TO_LABEL[metric]}}}"
                if j == 0
                else ""
            )
            lines.append(f"{head} & {name} & " + " & ".join(cells) + r" \\")
        lines.append(r"\bottomrule" if mi == len(METRICS) - 1 else r"\midrule")
    lines += [r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


In [ ]:
tex_dim_budget = dim_budget_table()
(PUB / "_dim_budget_table.tex").write_text(tex_dim_budget + "\n")
print(tex_dim_budget)
